## Module 4: Loss Functions and How Network Learns

In [7]:
# MSE in Pytorch
import torch
import torch.nn as nn

criterion = nn.MSELoss()
y_pred = torch.tensor([[2.5], [0.0], [2.0], [8.0]])
y_true = torch.tensor([[3.0], [-0.5], [2.0], [7.0]])
loss = criterion(y_pred, y_true)
print(f'MSE Loss: {loss.item():.4f}')

criterion_sum = nn.MSELoss(reduction = 'sum')
criterion_none = nn.MSELoss(reduction = 'none')

print('--' * 30)

# Binary Cross Entropy (BCE)
# Method 1: nn.BCELoss -- requires sigmoid already applied
y_pred_probs = torch.tensor([0.9, 0.2, 0.8, 0.1])
y_true = torch.tensor([1.0, 0.0, 1.0, 0.0])

criterion = nn.BCELoss()
loss = criterion(y_pred_probs, y_true)
print(f'BCE Loss: {loss.item():.4f}')

# Method 2: nn.BCEWithLogitsLoss -- takes raw logits, applies sigmoid internally
y_pred_logits = torch.tensor([2.2, -1.4, 1.9, -2.3])  # raw logits (before sigmoid)
y_true        = torch.tensor([1.0,  0.0, 1.0,  0.0])
criterion = nn.BCEWithLogitsLoss()
loss = criterion(y_pred_logits, y_true)
print(f'BCEWithLogits Loss: {loss.item():.4f}')

print('--' * 30)

MSE Loss: 0.3750
------------------------------------------------------------
BCE Loss: 0.1643
BCEWithLogits Loss: 0.1401
------------------------------------------------------------


In [8]:
# nn.CrossEntropyLoss
logits = torch.tensor([
    [1.5, 0.8, 3.2, 0.5],
    [0.1, 4.2, 0.3, 0.2],
    [2.0, 0.5, 0.5, 1.5],
])
labels = torch.tensor([2, 1, 0])
criterion = nn.CrossEntropyLoss()
loss = criterion(logits, labels)
print(f'Cross-Entropy Loss: {loss.item():.4f}')

Cross-Entropy Loss: 0.3553


In [9]:
# Backpropogation Automated
model = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 1), nn.Sigmoid())
with torch.no_grad():
    model[0].weight.copy_(torch.tensor([[0.1, 0.2], [0.3, 0.4]]))
    model[0].bias.fill_(0.0)
    model[2].weight.copy_(torch.tensor([[0.5, 0.6]]))
    model[2].bias.fill_(0.0)

x = torch.tensor([[2.0, 3.0]])
y = torch.tensor([[1.0]])

y_pred = model(x)
print(f'Prediction: {y_pred.item():.4f}')

criterion = nn.BCELoss()
loss = criterion(y_pred, y)
print(f'Loss: {loss.item():.4f}')

loss.backward()

print(f'\nGradients computed by PyTorch:')
print(f"W2 gradient: {model[2].weight.grad}")
print(f"b2 gradient: {model[2].bias.grad}")
print(f"W1 gradient: {model[0].weight.grad}")

Prediction: 0.8146
Loss: 0.2051

Gradients computed by PyTorch:
W2 gradient: tensor([[-0.1483, -0.3338]])
b2 gradient: tensor([-0.1854])
W1 gradient: tensor([[-0.1854, -0.2781],
        [-0.2225, -0.3338]])


In [10]:
# Optimizers
import torch.optim as optim

# SGD
optimizer = optim.SGD(model.parameters(), lr = 0.01, momentum = 0.9)

# Adam
optimizer = optim.Adam(model.parameters(), lr = 0.001, betas = (0.9, 0.999), eps = 1e-8)

# AdamW
optimizer = optim.AdamW(model.parameters(), lr = 0.001, weight_decay = 0.01)

In [11]:
# Complete Training Loop
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

x = torch.randn(1000, 10)
y = (x[:, 0] + x[:,  1] > 0).float().unsqueeze(1)

dataset = TensorDataset(x, y)
dataloader = DataLoader(dataset, batch_size = 32, shuffle = True)

class BinaryNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(10, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.net(x)

model = BinaryNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
num_epochs = 20

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    correct = 0
    total = 0

    for batch_X, batch_y in dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item() * batch_X.size(0)
        predicted = (torch.sigmoid(output) >= 0.5).float()
        correct += (predicted == batch_y).sum().item()
        total += batch_X.size(0)

    avg_loss = epoch_loss / total
    accuracy = correct / total * 100
    
    if (epoch + 1) % 5 == 0:
        print(f'Epoch: [{epoch+1:>3}/{num_epochs}] | Loss: {avg_loss:.4f} | Accuracy: {accuracy:.1f}%')

Epoch: [  5/20] | Loss: 0.1249 | Accuracy: 98.0%
Epoch: [ 10/20] | Loss: 0.0452 | Accuracy: 99.6%
Epoch: [ 15/20] | Loss: 0.0253 | Accuracy: 99.9%
Epoch: [ 20/20] | Loss: 0.0148 | Accuracy: 100.0%


In [ ]:
# Saving and Loading Models

# 1. Saving Only the Weights
torch.save(model.state_dict(), 'model_weights.pth')
model = BinaryNet()
model.load_state_dict(torch.load('model_weights.pth'))
model.eval()

# 2. Saving the Full Model
torch.save(model, 'full_model.pth')
model.eval()

In [ ]:
# Saving Training Checkpoints
checkpoints = {
    'epoch' : epoch,
    'model_state_dict' : model.state_dict(),
    'optimizer_state_dict' : optimizer.state_dict(),
    'loss' : loss
}
torch.save(checkpoints, f'checkpoint_epoch_{epoch}.pth')

checkpoint = torch.load(checkpoint_epoch_10.pth)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
start_epoch = checkpoint['epoch'] + 1

In [12]:
# Inference Using Trained Model
def predict(model, X_new, device):
    model.eval()
    X_new = X_new.to(device)

    with torch.no_grad():
        logits = model(X_new)
        probs = torch.sigmoid(logits)
        preds = (probs >= 0.5).int()

    return probs.cpu(), preds.cpu()

X_test = torch.randn(5, 10)
probs, preds = predict(model, X_test, device)
print(f'Probabilites : {probs.squeeze()}')
print(f'Predictions : {preds.squeeze()}')

Probabilites : tensor([1.9950e-05, 6.7073e-07, 9.8719e-10, 6.2124e-13, 5.3386e-13])
Predictions : tensor([0, 0, 0, 0, 0], dtype=torch.int32)


In [26]:
# Full Training and Validation Loop
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

x = torch.randn(1000, 10)
y = (x[:, 0] + x[:, 1] > 0.1).float().unsqueeze(1)

full_dataset = TensorDataset(x, y)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 32, shuffle = False)

model = BinaryNet().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)

history = {'train_loss' : [], 'val_loss' : [], 'train_acc' : [], 'val_acc' : []}

for epoch in range(30):
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * batch_X.size(0)
        predicted = (torch.sigmoid(output) >= 0.5).float()
        train_correct += (predicted == batch_y).sum().item()
        train_total += batch_X.size(0)

    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            output = model(batch_X)
            loss = criterion(output, batch_y)

            val_loss += loss.item() * batch_X.size(0)
            predicted = (torch.sigmoid(output) >= 0.5).float()
            val_correct += (predicted == batch_y).sum().item()
            val_total += batch_X.size(0)


    t_loss = train_loss / train_total
    v_loss = val_loss / val_total
    t_acc = train_correct / train_total * 100
    v_acc = val_correct / val_total * 100

    history['train_loss'].append(t_loss)
    history['val_loss'].append(v_loss)
    history['train_acc'].append(t_acc)
    history['val_acc'].append(v_acc)

    if (epoch + 1) % 5 == 0:
        print(f"Epoch: {epoch + 1} | Train Loss: {t_loss:.4f} | Val Loss {v_loss:.4f} | Train Acc: {t_acc}% | Val Acc: {v_acc}%")

torch.save(model.state_dict(), 'binary_model.pth')
print(f'\nModel Saved to binary_model.pth')

Epoch: 5 | Train Loss: 0.1929 | Val Loss 0.1453 | Train Acc: 96.0% | Val Acc: 98.5%
Epoch: 10 | Train Loss: 0.0667 | Val Loss 0.0632 | Train Acc: 99.25% | Val Acc: 98.5%
Epoch: 15 | Train Loss: 0.0355 | Val Loss 0.0476 | Train Acc: 99.75% | Val Acc: 98.0%
Epoch: 20 | Train Loss: 0.0211 | Val Loss 0.0471 | Train Acc: 99.875% | Val Acc: 98.5%
Epoch: 25 | Train Loss: 0.0134 | Val Loss 0.0486 | Train Acc: 100.0% | Val Acc: 98.0%
Epoch: 30 | Train Loss: 0.0088 | Val Loss 0.0520 | Train Acc: 100.0% | Val Acc: 98.0%

Model Saved to binary_model.pth


In [36]:
# Mini Exercise = Building a complete training pipeline for multiclass classification on Iris Dataset.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
import numpy as np

torch.manual_seed(42)
np.random.seed(42)

iris = load_iris()
x = iris.data
y = iris.target

scaler = StandardScaler()
x = scaler.fit_transform(x)

x = torch.tensor(x, dtype = torch.float32)
y = torch.tensor(y, dtype = torch.long)
print(f"Input Shape: {x.shape}")
print(f"Target Shape{y.shape}")

dataset = TensorDataset(x, y)
train_ds, val_ds = random_split(dataset, [120, 30])

train_loader = DataLoader(train_ds, batch_size = 16, shuffle = True)
val_loader = DataLoader(val_ds, batch_size = 16, shuffle = False)

class IrisNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 16), nn.ReLU(), nn.Linear(16, 8), nn.ReLU(), nn.Linear(8, 3)
        )

    def forward(self, x):
        x = self.net(x)
        return x

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = IrisNet().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

history = {'train_loss' : [], 'val_loss' : [], 'val_acc' : []}

for epoch in range(100):
    model.train()
    total_loss = 0.0

    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)

        optimizer.zero_grad()
        output = model(batch_X)
        loss = criterion(output, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_X.size(0)

    avg_train_loss = total_loss / len(train_ds)

    model.eval()
    val_loss, correct, total = 0.0, 0, 0

    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)

            output = model(batch_X)
            loss = criterion(output, batch_y)

            val_loss += loss.item() * batch_X.size(0)
            preds = torch.argmax(output, dim = 1)
            correct += (preds == batch_y).sum().float()
            total += batch_X.size(0)

    avg_val_loss = val_loss / len(val_ds)
    val_acc = correct / total * 100

    history['train_loss'].append(avg_train_loss)
    history['val_loss'].append(avg_val_loss)
    history['val_acc'].append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch: {epoch + 1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.2f}%")

torch.save(model.state_dict(), 'iris_model.pth')
print('\nModel Saved as iris_model.pth')

model.eval()
sample = torch.tensor([[5.1, 3.5, 1.4, 0.2]], dtype = torch.float32).to(device)
sample_scaled = torch.tensor(scaler.transform([[5.1, 3.5, 1.4, 0.2]]), dtype = torch.float32).to(device)

with torch.no_grad():
    logits = model(sample_scaled)
    probs = torch.softmax(logits, dim = 1)
    pred = torch.argmax(probs, dim = 1)

class_names = iris.target_names
print(f'\nSample: {sample}')
print(f'Probabilities: {probs.cpu().numpy().round(3)}')
print(f'Predicted Class: {class_names[pred.item()]}')

Input Shape: torch.Size([150, 4])
Target Shapetorch.Size([150])
Epoch: 10 | Train Loss: 0.0983 | Val Loss: 0.1765 | Val Acc: 90.00%
Epoch: 20 | Train Loss: 0.0257 | Val Loss: 0.1488 | Val Acc: 93.33%
Epoch: 30 | Train Loss: 0.0136 | Val Loss: 0.1948 | Val Acc: 93.33%
Epoch: 40 | Train Loss: 0.0091 | Val Loss: 0.2219 | Val Acc: 96.67%
Epoch: 50 | Train Loss: 0.0094 | Val Loss: 0.2406 | Val Acc: 93.33%
Epoch: 60 | Train Loss: 0.0040 | Val Loss: 0.2260 | Val Acc: 96.67%
Epoch: 70 | Train Loss: 0.0029 | Val Loss: 0.2585 | Val Acc: 96.67%
Epoch: 80 | Train Loss: 0.0020 | Val Loss: 0.2853 | Val Acc: 96.67%
Epoch: 90 | Train Loss: 0.0013 | Val Loss: 0.3210 | Val Acc: 90.00%
Epoch: 100 | Train Loss: 0.0010 | Val Loss: 0.3269 | Val Acc: 93.33%

Model Saved as iris_model.pth

Sample: tensor([[5.1000, 3.5000, 1.4000, 0.2000]], device='cuda:0')
Probabilities: [[1. 0. 0.]]
Predicted Class: setosa
